In [ ]:
# ═══════════ CONFIGURATION ═══════════
REPO_URL = "https://github.com/ASAYMAN69/colab.git"   # clone source for the server
SERVER_PORT = 8787                                    # local port
BRIDGE_TOKEN = ""                                     # optional; empty = NO AUTH
# ═════════════════════════════════════════

import glob, json, os, platform, re, shutil, stat, subprocess, sys, threading, time, urllib.request

# ---- 1. make server.py available (clone if we are not inside the repo) ----
HERE = os.getcwd()
SERVER_PY = os.path.join(HERE, "server.py")
if not os.path.exists(SERVER_PY):
    if not REPO_URL:
        raise SystemExit("server.py not found and REPO_URL is empty")
    get_ipython().system(f"git clone --depth 1 {REPO_URL} _src")
    matches = glob.glob(os.path.join("_src", "**", "server.py"), recursive=True)
    if not matches:
        raise SystemExit("server.py not found inside the cloned repo")
    shutil.copy(matches[0], HERE)

# ---- 2. start the HTTP server ----
env = dict(os.environ)
if BRIDGE_TOKEN:
    env["BRIDGE_TOKEN"] = BRIDGE_TOKEN
proc = subprocess.Popen(
    [sys.executable, "server.py", "--port", str(SERVER_PORT)],
    env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
ready = False
for _ in range(50):
    try:
        with urllib.request.urlopen(
            f"http://127.0.0.1:{SERVER_PORT}/status", timeout=1
        ) as r:
            print("server online:", json.load(r))
            ready = True
            break
    except Exception:
        time.sleep(0.2)
if not ready:
    raise SystemExit("server failed to start:\n" + (proc.stdout.read() if proc.stdout else ""))

# ---- 3. ensure cloudflared (download if missing) ----
CF = shutil.which("cloudflared") or os.path.expanduser("~/.local/bin/cloudflared")
if not os.path.exists(CF):
    tag = {
        "x86_64": "amd64", "AMD64": "amd64",
        "aarch64": "arm64", "arm64": "arm64",
    }.get(platform.machine())
    if not tag:
        raise SystemExit("unsupported CPU architecture: " + platform.machine())
    os.makedirs(os.path.dirname(CF), exist_ok=True)
    url = ("https://github.com/cloudflare/cloudflared/releases/latest/download/"
           f"cloudflared-linux-{tag}")
    print("downloading cloudflared:", url)
    urllib.request.urlretrieve(url, CF)
    os.chmod(CF, os.stat(CF).st_mode | stat.S_IXUSR)
print("cloudflared ready:", CF)

# ---- 4. open the Cloudflare tunnel and hand out the URL ----
tunnel = subprocess.Popen(
    [CF, "tunnel", "--url", f"http://127.0.0.1:{SERVER_PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
URL_RE = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
url, logs = None, []

def reader():
    global url
    for line in iter(tunnel.stdout.readline, ""):
        logs.append(line.rstrip())
        m = URL_RE.search(line)
        if m:
            url = m.group(0)

threading.Thread(target=reader, daemon=True).start()

deadline = time.time() + 90
while time.time() < deadline and url is None:
    time.sleep(0.5)

if url:
    with open("cloudflare_url.txt", "w") as fh:
        fh.write(url + "\n")
    print("\n" + "=" * 68)
    print("PUBLIC URL -> " + url)
    print("=" * 68)
    print("Paste this URL to the AI. Keep this notebook running; stop it when done.")
else:
    print("No Cloudflare URL within 90s. Last tunnel logs:")
    print("\n".join(logs[-15:]))
    print("Tip: quick tunnels are sometimes rate-limited - wait a minute and rerun this cell.")

In [ ]:
# Stop server + tunnel so the public URL closes.
stopped = []
for p, name in ((locals().get("proc"), "server"), (locals().get("tunnel"), "tunnel")):
    if p is not None and p.poll() is None:
        p.terminate()
        stopped.append(name)
print("stopped:", stopped or "nothing running")